# Industry pipeline

Links every asset to its issuer in EDGAR and provides the **Fama-French industry definitions** of all schemes
(5, 10, 12, 17, 30, 38, 48, 49), so the classification is chosen at query time:

* each asset is linked to its issuer's CIK by name (confirmed by ticker) in the EDGAR company tables
  `clean_up_1.edgar_company` + `clean_up_1.edgar_company_name` (built by `4_owner_pipeline.ipynb`) →
  `input_1.asset_issuer_match` → `issuer_cik`, `industry_match` on `clean_up_1.asset`
* Kenneth French's definitions → `clean_up_1.ff_industry` (industries) + `clean_up_1.sic_ff` (SIC → industry per
  scheme)

```sql
-- industry of an asset in scheme 12 (any scheme: change f.scheme)
FROM clean_up_1.asset a
JOIN clean_up_1.edgar_company c ON c.cik = a.issuer_cik
JOIN clean_up_1.sic_ff f        ON f.sic = c.sic AND f.scheme = 12
JOIN clean_up_1.ff_industry i   USING (scheme, industry_number)
```

Every block shows the database feedback and is written to the run log `logs/industry/<RUN_ID>/`, so runs can be
compared later (section 5). Execution and logging code lives in `src/pipeline_helpers.py`.

| Section | Phase label | What happens |
|---|---|---|
| 0.2 Database connection | `connection` | read-only check, then working connection |
| 1. FF industry definitions | `industry_ff` | `ff_industry`, `sic_ff` for all FF schemes |
| 2. Issuer matching | `industry_match` | asset issuer name → EDGAR CIK, ticker as confirmation |
| 3. Asset columns | `industry_asset` | `issuer_cik`, `industry_match` on `clean_up_1.asset` |
| 4. Checks | `industry_checks` | coverage of the assets and the holdings (per scheme), conflicts, samples |

**Limitations**
* EDGAR has the **current** SIC of a filer, not its history. A firm that changed industry (e.g. a SPAC after its
  merger, SIC 6770 → operating SIC) is classified by today's code.
* Funds and ETFs have no SIC in EDGAR, so they stay unclassified.
* `sic_ff` follows Ken French's definitions exactly: schemes 17, 30, 48 and 49 have no catch-all industry, so a SIC
  they do not list is not classified by them (schemes 5, 10, 12 and 38 classify every SIC).
* Tickers in EDGAR are today's tickers only. They are used to confirm a name match, never alone: a ticker of a
  delisted firm is often reused (Nextel `S` → SentinelOne) or the OpenFIGI ticker is the acquirer's (AirTran bond
  → `LUV`).

> **The cells modify the database.** `ff_industry`, `sic_ff` and the match table are dropped and rebuilt; the
> issuer columns of `clean_up_1.asset` are reset and refilled (the other asset columns are not touched).

## 0. Setup

### 0.1 Configuration

* `SICCODES_DIR`: Kenneth French's industry definitions as CSV (`sic_to_ff_lookup.csv`, `ff_industries.csv`;
  source: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_48_ind_port.html)
* `CHECK_SCHEME`: the FF scheme the checks in section 4 report on (the tables hold all schemes)

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent                       # .../Database/Initialisation (notebooks run from notebooks/)
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "config")]

from pipeline_helpers import (IssuerMatcher, PipelineRun, compare_runs,
                              expected_mismatches, load_logs, phase_durations, steps_table)
from local_config import DSN, SICCODES_DIR   # machine-specific, git-ignored (template: local_config.example.py)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

LOG_ROOT = ROOT / "logs" / "industry"

COMPANY_TABLE  = "clean_up_1.edgar_company"        # built by 4_owner_pipeline.ipynb
NAME_TABLE     = "clean_up_1.edgar_company_name"
FF_INDUSTRY_TABLE = "clean_up_1.ff_industry"
SIC_FF_TABLE   = "clean_up_1.sic_ff"
MATCH_TABLE    = "input_1.asset_issuer_match"
ASSET_TABLE    = "clean_up_1.asset"
HOLDINGS_TABLE = "clean_up_1.holdings_clean"

CHECK_SCHEME = 48   # FF scheme used by the checks (5, 10, 12, 17, 30, 38, 48 or 49)

# Replaced by ff_industry / sic_ff (earlier version of this pipeline), dropped in section 1
REPLACED_TABLES = ["input_1.sic_to_ff", "input_1.ff_industries"]

p = PipelineRun(LOG_ROOT, dsn=DSN, max_rows=1000, exact_counts=True)
print("RUN_ID :", p.run_id)
print("LOG_DIR:", p.log_dir)

RUN_ID : 20261029_160418
LOG_DIR: <THESIS_DIR>/src/Clean_solution/Database/Initialisation/logs/industry/20261029_160418


### 0.2 Database connection

Read-only check first (`READ ONLY` transaction, rolled back). Required tables: `clean_up_1.asset` (built by
`2_f13_securities_pipeline.ipynb`, tickers from `3_openfigi_pipeline.ipynb`) and `clean_up_1.edgar_company` /
`clean_up_1.edgar_company_name` (built by `4_owner_pipeline.ipynb`).

In [2]:
CONNECTION_OK = p.check_connection(DSN, schemas=["input_1", "clean_up_1"],
                                   tables=[COMPANY_TABLE, NAME_TABLE, FF_INDUSTRY_TABLE, SIC_FF_TABLE, MATCH_TABLE,
                                           ASSET_TABLE, HOLDINGS_TABLE])

✅ **[connection] connection_check_server**: `ok` | 0.0s

> READ ONLY transaction, rolled back. Nothing committed.

,server_version,database,db_user,is_standby,txn_read_only,db_size,other_sessions
0,PostgreSQL 16.15 (Ubuntu 16.15-0ubuntu0.24.04....,Holdings_v2,postgres,False,on,70 GB,0


✅ **[connection] connection_check_schemas**: `ok`

,schema_name,schema_exists,can_create,can_use
0,input_1,True,True,True
1,clean_up_1,True,True,True


✅ **[connection] connection_check_tables**: `ok`

> Existing tables will be dropped and recreated by the pipeline.

,table_name,exists,unlogged,est_rows,total_size
0,clean_up_1.edgar_company,True,False,88009,19 MB
1,clean_up_1.edgar_company_name,True,False,118248,14 MB
2,clean_up_1.asset,True,False,40593,20 MB
3,input_1.asset_issuer_match,True,False,40593,4928 kB
4,clean_up_1.ff_industry,True,False,209,64 kB
5,clean_up_1.sic_ff,True,False,59627,4912 kB
6,clean_up_1.holdings_clean,True,False,97681472,25 GB


In [3]:
assert CONNECTION_OK, "Connection check failed, see connection_check_server above"
p.connect()

Connected: user=postgres password=xxx dbname=Holdings_v2 host=localhost port=5432


## 1. Fama-French industry definitions

Kenneth French's SIC definitions as CSV (`SICCODES_DIR`), two tables:

* `clean_up_1.ff_industry`: one row per industry of every scheme (5, 10, 12, 17, 30, 38, 48, 49): number, short and
  long name, `residual_category` (the scheme's catch-all *Other*, which takes every SIC not listed elsewhere; only
  schemes 5, 10, 12 and 38 have one)
* `clean_up_1.sic_ff`: one row per SIC code and scheme that classifies it (`sic` as 4 characters, like
  `edgar_company.sic`). Built from `sic_to_ff_lookup.csv` (one column per scheme), blanks dropped: a SIC without a
  row in a scheme is not classified by that scheme (FF48 lists 5,054 of the 10,000 codes)

The `*_sic_ranges.csv` files and the name columns of the lookup repeat this information and are not loaded.

In [4]:
ff = p.stage("industry_ff")

ff.run("create_ff_tables", f"""
    DROP TABLE IF EXISTS {SIC_FF_TABLE};
    DROP TABLE IF EXISTS {FF_INDUSTRY_TABLE};

    CREATE TABLE {FF_INDUSTRY_TABLE} (
        scheme             SMALLINT NOT NULL CHECK (scheme IN (5, 10, 12, 17, 30, 38, 48, 49)),
        industry_number    SMALLINT NOT NULL,
        short_name         TEXT     NOT NULL,
        industry_name      TEXT     NOT NULL,
        residual_category  BOOLEAN  NOT NULL,
        PRIMARY KEY (scheme, industry_number)
    );

    CREATE TABLE {SIC_FF_TABLE} (
        sic              VARCHAR(4) NOT NULL CHECK (sic ~ '^[0-9]{{4}}$'),
        scheme           SMALLINT   NOT NULL,
        industry_number  SMALLINT   NOT NULL,
        PRIMARY KEY (sic, scheme),
        FOREIGN KEY (scheme, industry_number) REFERENCES {FF_INDUSTRY_TABLE} (scheme, industry_number)
    );
""")

ff.copy_csv("import_ff_industry", FF_INDUSTRY_TABLE, SICCODES_DIR / "ff_industries.csv",
            ["scheme", "industry_number", "short_name", "industry_name", "residual_category"],
            expected=209)

# lookup (one column per scheme) -> one row per (sic, scheme); the long file goes to the run log
lookup = pd.read_csv(SICCODES_DIR / "sic_to_ff_lookup.csv", dtype=str)
schemes = [c for c in lookup.columns if c.startswith("ff") and not c.endswith("_name")]
sic_ff = (lookup.melt(id_vars="sic", value_vars=schemes, var_name="scheme", value_name="industry_number")
                .dropna(subset=["industry_number"]))
sic_ff["scheme"] = sic_ff["scheme"].str[2:]
sic_ff_csv = p.log_dir / "sic_ff.csv"
sic_ff.sort_values(["scheme", "sic"]).to_csv(sic_ff_csv, index=False)
ff.copy_csv("import_sic_ff", SIC_FF_TABLE, sic_ff_csv, ["sic", "scheme", "industry_number"],
            expected=len(sic_ff), note=f"{len(lookup):,} SIC codes x {len(schemes)} schemes, blanks dropped")

ff.run("idx_sic_ff_industry", f"CREATE INDEX IF NOT EXISTS sic_ff_industry_idx ON {SIC_FF_TABLE} (scheme, industry_number);")
ff.run("analyze_ff", f"ANALYZE {FF_INDUSTRY_TABLE}; ANALYZE {SIC_FF_TABLE};")

✅ **[industry_ff] create_ff_tables**: `ok` | `CREATE TABLE` | 0.0s

✅ **[industry_ff] import_ff_industry**: `ok` | `COPY 209` | rows: **209** | expected: 209 (diff +0) | 0.0s

✅ **[industry_ff] import_sic_ff**: `ok` | `COPY 59627` | rows: **59,627** | expected: 59,627 (diff +0) | 0.2s

> 10,000 SIC codes x 8 schemes, blanks dropped

✅ **[industry_ff] idx_sic_ff_industry**: `ok` | `CREATE INDEX` | 0.0s

✅ **[industry_ff] analyze_ff**: `ok` | `ANALYZE` | 0.0s

""


In [5]:
ff.run("ff_schemes", f"""
    -- industries per scheme and how many SIC codes each scheme classifies
    SELECT i.scheme, count(*) AS industries, bool_or(i.residual_category) AS has_residual,
           (SELECT count(*) FROM {SIC_FF_TABLE} f WHERE f.scheme = i.scheme) AS sic_codes
    FROM {FF_INDUSTRY_TABLE} i
    GROUP BY i.scheme
    ORDER BY i.scheme;
""", note="Previous test: 5 / 10 / 12 / 38 all 10,000 SIC codes | 17: 4,529 | 30: 4,990 | 48 and 49: 5,054")

ff.run("ff_sanity", f"""
    SELECT f.sic, f.scheme, f.industry_number, i.short_name
    FROM {SIC_FF_TABLE} f
    JOIN {FF_INDUSTRY_TABLE} i USING (scheme, industry_number)
    WHERE f.sic IN ('6021', '3674', '7372', '9999') AND f.scheme IN (12, 48)
    ORDER BY f.sic, f.scheme;
""", note="expected FF48: 6021 Banks (44) | 3674 Chips (36) | 7372 BusSv (34) | 9999 not listed; "
          "FF12: 6021 Money | 3674 BusEq | 7372 BusEq | 9999 Other")

✅ **[industry_ff] ff_schemes**: `ok` | `SELECT 8` | rows: **8** | 0.0s

> Previous test: 5 / 10 / 12 / 38 all 10,000 SIC codes | 17: 4,529 | 30: 4,990 | 48 and 49: 5,054

,scheme,industries,has_residual,sic_codes
0,5,5,True,10000
1,10,10,True,10000
2,12,12,True,10000
3,17,17,False,4529
4,30,30,False,4990
5,38,38,True,10000
6,48,48,False,5054
7,49,49,False,5054


✅ **[industry_ff] ff_sanity**: `ok` | `SELECT 7` | rows: **7** | 0.0s

> expected FF48: 6021 Banks (44) | 3674 Chips (36) | 7372 BusSv (34) | 9999 not listed; FF12: 6021 Money | 3674 BusEq | 7372 BusEq | 9999 Other

,sic,scheme,industry_number,short_name
0,3674,12,6,BusEq
1,3674,48,36,Chips
2,6021,12,11,Money
3,6021,48,44,Banks
4,7372,12,6,BusEq
5,7372,48,34,BusSv
6,9999,12,12,Other


,sic,scheme,industry_number,short_name
0,3674,12,6,BusEq
1,3674,48,36,Chips
2,6021,12,11,Money
3,6021,48,44,Banks
4,7372,12,6,BusEq
5,7372,48,34,BusSv
6,9999,12,12,Other


In [6]:
for t in REPLACED_TABLES:
    ff.run(f"drop_{t.split('.')[1]}", f"DROP TABLE IF EXISTS {t};")
ff.run("drop_ff48_of", "DROP FUNCTION IF EXISTS input_1.ff48_of(integer);")

✅ **[industry_ff] drop_sic_to_ff**: `ok` | `DROP TABLE` | 0.0s

> NOTICES: NOTICE:  table "sic_to_ff" does not exist, skipping

✅ **[industry_ff] drop_ff_industries**: `ok` | `DROP TABLE` | 0.0s

> NOTICES: NOTICE:  table "ff_industries" does not exist, skipping

✅ **[industry_ff] drop_ff48_of**: `ok` | `DROP FUNCTION` | 0.0s

> NOTICES: NOTICE:  function input_1.ff48_of(pg_catalog.int4) does not exist, skipping

""


## 2. Issuer matching

`input_1.asset_issuer_match`: one row per asset with the issuer's CIK and how it was found
(`pipeline_helpers.IssuerMatcher`):

1. **Candidates** are EDGAR filers with a SIC whose current or former name matches the asset's issuer name word by
   word. A word matches if equal, or if one is an abbreviation of the other (same first letter, prefix or
   letters in order: `TECH` ~ `TECHNOLOGIES`, `ELECTRS` ~ `ELECTRONICS`, `PWR` ~ `POWER`). 13F issuer names
   are cut at ~30 characters, so names of 28+ characters may stop early. A one-word name must match exactly.
2. **Filters:** an exact name match beats abbreviation matches (`CARS COM` → Cars.com, not CarPartsOnSale.com);
   an operating filer beats ADR registrants (SIC 8880) and shells (SIC 9995) of the same name (Teva, Syngenta).
3. **One CIK** is picked by the first rule that leaves exactly one candidate:

| `method` | rule |
|---|---|
| `name+ticker` | a candidate also has the asset's OpenFIGI ticker (first word, so bonds count too) in EDGAR |
| `name` | only one candidate |
| `name+period` | one candidate used the matched name while the asset was listed (`first_year`–`last_year`) |
| `name+same_industry` | all candidates are in the same FF48 industry (or have the same SIC); the one whose name was used last is kept |
| `ambiguous` | candidates in different industries → no CIK |
| `unmatched` | no candidate |

A former name links to the filer's **current** CIK: Theravance → Innoviva, A.T. Cross → Costa, FireEye →
Mandiant are the same filer renamed.

In [7]:
mat = p.stage("industry_match")

mat.run("create_match_table", f"""
    DROP TABLE IF EXISTS {MATCH_TABLE};

    CREATE TABLE {MATCH_TABLE} (
        cusip           VARCHAR(9) PRIMARY KEY,
        cik             INTEGER,
        method          TEXT       NOT NULL CHECK (method IN ('name+ticker', 'name', 'name+period',
                                                              'name+same_industry', 'ambiguous', 'unmatched')),
        candidates      SMALLINT   NOT NULL,
        candidate_ciks  INTEGER[]  NOT NULL,
        ticker_cik      INTEGER,
        CHECK ((cik IS NULL) = (method IN ('ambiguous', 'unmatched')))
    );
""")

✅ **[industry_match] create_match_table**: `ok` | `CREATE TABLE` | 0.0s

""


In [8]:
matcher = IssuerMatcher(mat, ASSET_TABLE, COMPANY_TABLE, NAME_TABLE, SIC_FF_TABLE, MATCH_TABLE)
matcher.run()
# previous test: name 25,461 | name+ticker 4,308 | name+same_industry 1,413 | name+period 498 |
#                ambiguous 662 | unmatched 8,251 (mostly funds / ETFs: no SIC in EDGAR)

✅ **[industry_match] match_issuers**: `ok` | rows: **40,593** | 3.8s

> 40,593 assets | 92,696 EDGAR names with SIC | 9,251 EDGAR tickers

,method,assets
0,name,25461
1,unmatched,8251
2,name+ticker,4308
3,name+same_industry,1413
4,ambiguous,662
5,name+period,498


,method,assets
0,name,25461
1,unmatched,8251
2,name+ticker,4308
3,name+same_industry,1413
4,ambiguous,662
5,name+period,498


In [9]:
mat.row_estimate(MATCH_TABLE, "match")
mat.run("match_by_security_type", f"""
    SELECT coalesce(a.security_type2, '(no FIGI)')                         AS security_type2,
           count(*)                                                        AS assets,
           count(m.cik)                                                    AS matched,
           round(100.0 * count(m.cik) / count(*), 1)                       AS matched_pct,
           count(*) FILTER (WHERE m.method = 'name+ticker')                AS by_ticker,
           count(*) FILTER (WHERE m.method = 'ambiguous')                  AS ambiguous
    FROM {MATCH_TABLE} m
    JOIN {ASSET_TABLE} a USING (cusip)
    GROUP BY 1
    ORDER BY 2 DESC;
""")

✅ **[industry_match] match_count_exact**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,n
0,40593


✅ **[industry_match] match_by_security_type**: `ok` | `SELECT 12` | rows: **12** | 0.0s

,security_type2,assets,matched,matched_pct,by_ticker,ambiguous
0,(no FIGI),27381,21677,79.2,0,526
1,Common Stock,4882,4639,95.0,2516,34
2,Corp,4152,3767,90.7,1140,67
3,Mutual Fund,2504,96,3.8,45,1
4,Preferred Stock,836,749,89.6,237,11
5,Depositary Receipt,428,386,90.2,208,1
6,REIT,210,195,92.9,123,5
7,Partnership Shares,78,71,91.0,29,1
8,Unit,67,52,77.6,3,14
9,Warrant,44,40,90.9,7,2


,security_type2,assets,matched,matched_pct,by_ticker,ambiguous
0,(no FIGI),27381,21677,79.2,0,526
1,Common Stock,4882,4639,95.0,2516,34
2,Corp,4152,3767,90.7,1140,67
3,Mutual Fund,2504,96,3.8,45,1
4,Preferred Stock,836,749,89.6,237,11
5,Depositary Receipt,428,386,90.2,208,1
6,REIT,210,195,92.9,123,5
7,Partnership Shares,78,71,91.0,29,1
8,Unit,67,52,77.6,3,14
9,Warrant,44,40,90.9,7,2


## 3. Asset columns

Columns on `clean_up_1.asset` (reset to `NULL` first, so a re-run is clean):

* `issuer_cik`: EDGAR CIK of the issuer; its SIC is `edgar_company.sic`, its industry in any scheme comes from
  `sic_ff` (see the query at the top)
* `industry_match`: the matching method from section 2

`sic`, `ff48` and `ff48_short` of the earlier version are dropped (the industry is no longer stored per asset).

In [10]:
col = p.stage("industry_asset")

col.run("asset_columns", f"""
    ALTER TABLE {ASSET_TABLE}
        DROP COLUMN IF EXISTS sic,
        DROP COLUMN IF EXISTS ff48,
        DROP COLUMN IF EXISTS ff48_short,
        ADD COLUMN IF NOT EXISTS issuer_cik      INTEGER,
        ADD COLUMN IF NOT EXISTS industry_match  TEXT;
""")

col.run("reset_issuer_columns", f"""
    UPDATE {ASSET_TABLE}
    SET issuer_cik = NULL, industry_match = NULL;
""", expected=40_593, note="all asset rows")

✅ **[industry_asset] asset_columns**: `ok` | `ALTER TABLE` | 0.0s

> NOTICES: NOTICE:  column "sic" of relation "asset" does not exist, skipping NOTICE:  column "ff48" of relation "asset" does not exist, skipping NOTICE:  column "ff48_short" of relation "asset" does not exist, skipping NOTICE:  column "issuer_cik" of relation "asset" already exists, skipping NOTICE:  column "industry_match" of relation "asset" already exists, skipping

✅ **[industry_asset] reset_issuer_columns**: `ok` | `UPDATE 40593` | rows: **40,593** | expected: 40,593 (diff +0) | 0.5s

> all asset rows

""


In [11]:
col.run("update_issuer", f"""
    UPDATE {ASSET_TABLE} a
    SET issuer_cik     = m.cik,
        industry_match = m.method
    FROM {MATCH_TABLE} m
    WHERE a.cusip = m.cusip;
""", expected=40_593, note="every asset (unmatched ones get industry_match only)")

col.run("idx_asset_issuer_cik", f"CREATE INDEX IF NOT EXISTS asset_issuer_cik_idx ON {ASSET_TABLE} (issuer_cik);")
col.run("analyze_asset", f"ANALYZE {ASSET_TABLE};")

✅ **[industry_asset] update_issuer**: `ok` | `UPDATE 40593` | rows: **40,593** | expected: 40,593 (diff +0) | 0.4s

> every asset (unmatched ones get industry_match only)

✅ **[industry_asset] idx_asset_issuer_cik**: `ok` | `CREATE INDEX` | 0.0s

> NOTICES: NOTICE:  relation "asset_issuer_cik_idx" already exists, skipping

✅ **[industry_asset] analyze_asset**: `ok` | `ANALYZE` | 0.1s

""


## 4. Checks

The checks classify through `asset.issuer_cik → edgar_company.sic → sic_ff` in `CHECK_SCHEME`
(`ASSET_INDUSTRY` below: one row per asset with its SIC and industry in that scheme).

### Coverage of the assets

In [12]:
chk = p.stage("industry_checks")

ASSET_INDUSTRY = f"""(
    SELECT a.*, c.sic, f.industry_number AS ff, i.short_name AS ff_short
    FROM {ASSET_TABLE} a
    LEFT JOIN {COMPANY_TABLE}     c ON c.cik = a.issuer_cik
    LEFT JOIN {SIC_FF_TABLE}      f ON f.sic = c.sic AND f.scheme = {CHECK_SCHEME}
    LEFT JOIN {FF_INDUSTRY_TABLE} i ON i.scheme = f.scheme AND i.industry_number = f.industry_number
)"""

chk.run("asset_coverage", f"""
    SELECT industry_match, count(*) AS assets, count(sic) AS with_sic, count(ff) AS with_industry
    FROM {ASSET_INDUSTRY} a
    GROUP BY 1
    ORDER BY 2 DESC;
""", note=f"with_industry: classified in FF{CHECK_SCHEME}")

chk.run("industry_distribution_assets", f"""
    SELECT ff, ff_short, count(*) AS assets
    FROM {ASSET_INDUSTRY} a
    WHERE ff IS NOT NULL
    GROUP BY 1, 2
    ORDER BY 1;
""", note=f"FF{CHECK_SCHEME}")

chk.run("unlisted_sic", f"""
    -- issuer SIC codes that CHECK_SCHEME does not list (assets with a SIC but no industry)
    SELECT a.sic, c.sic_description, count(*) AS assets
    FROM {ASSET_INDUSTRY} a
    LEFT JOIN (SELECT DISTINCT sic, sic_description FROM {COMPANY_TABLE}) c USING (sic)
    WHERE a.sic IS NOT NULL AND a.ff IS NULL
    GROUP BY 1, 2
    ORDER BY 3 DESC;
""", note=f"FF{CHECK_SCHEME}; schemes 5, 10, 12 and 38 classify every SIC")

✅ **[industry_checks] asset_coverage**: `ok` | `SELECT 6` | rows: **6** | 0.0s

> with_industry: classified in FF48

,industry_match,assets,with_sic,with_industry
0,name,25461,25461,25341
1,unmatched,8251,0,0
2,name+ticker,4308,4308,4301
3,name+same_industry,1413,1413,1411
4,ambiguous,662,0,0
5,name+period,498,498,495


✅ **[industry_checks] industry_distribution_assets**: `ok` | `SELECT 48` | rows: **48** | 0.0s

> FF48

,ff,ff_short,assets
0,1,Agric,89
1,2,Food,308
2,3,Soda,29
3,4,Beer,73
4,5,Smoke,38
5,6,Toys,202
6,7,Fun,459
7,8,Books,150
8,9,Hshld,287
9,10,Clths,242


✅ **[industry_checks] unlisted_sic**: `ok` | `SELECT 4` | rows: **4** | 0.0s

> FF48; schemes 5, 10, 12 and 38 classify every SIC

,sic,sic_description,assets
0,3990,Miscellaneous Manufacturing Industries,65
1,9995,Non-Operating Establishments,37
2,6189,Asset-Backed Securities,21
3,0900,"Fishing, Hunting and Trapping",9


,sic,sic_description,assets
0,3990,Miscellaneous Manufacturing Industries,65
1,9995,Non-Operating Establishments,37
2,6189,Asset-Backed Securities,21
3,0900,"Fishing, Hunting and Trapping",9


### Coverage of the holdings

What matters for the analysis is the share of **holdings** that get an industry, not of listed CUSIPs. The
holdings are aggregated per CUSIP and year once (temporary table, ~1–2 minutes over ~100M rows).
`scheme_coverage` compares all schemes; the other checks use `CHECK_SCHEME`.

In [13]:
chk.run("holdings_per_cusip_year", f"""
    DROP TABLE IF EXISTS pg_temp.holdings_cusip_year;
    CREATE TEMP TABLE holdings_cusip_year AS
    SELECT cusip9, extract(year FROM rdate)::int AS year, count(*) AS rows, sum(value) AS value
    FROM {HOLDINGS_TABLE}
    GROUP BY 1, 2;
""")

chk.run("scheme_coverage", f"""
    -- share of the holdings classified by each scheme
    SELECT s.scheme,
           round(100.0 * sum(h.rows)  FILTER (WHERE f.sic IS NOT NULL) / sum(h.rows), 1)  AS rows_pct,
           round(100.0 * sum(h.value) FILTER (WHERE f.sic IS NOT NULL) / sum(h.value), 1) AS value_pct
    FROM pg_temp.holdings_cusip_year h
    LEFT JOIN {ASSET_TABLE}   a ON a.cusip = h.cusip9
    LEFT JOIN {COMPANY_TABLE} c ON c.cik = a.issuer_cik
    CROSS JOIN (SELECT DISTINCT scheme FROM {FF_INDUSTRY_TABLE}) s
    LEFT JOIN {SIC_FF_TABLE}  f ON f.sic = c.sic AND f.scheme = s.scheme
    GROUP BY s.scheme
    ORDER BY s.scheme;
""", note="schemes with a catch-all industry (5, 10, 12, 38) classify every holding whose issuer has a SIC")

chk.run("holdings_coverage_per_year", f"""
    SELECT h.year,
           sum(h.rows)                                                                    AS rows,
           round(100.0 * sum(h.rows)  FILTER (WHERE a.ff IS NOT NULL) / sum(h.rows), 1)    AS rows_pct,
           round(100.0 * sum(h.value) FILTER (WHERE a.ff IS NOT NULL) / sum(h.value), 1)   AS value_pct,
           round(100.0 * sum(h.value) FILTER (WHERE a.cusip IS NULL)  / sum(h.value), 1)   AS value_not_in_asset_pct
    FROM pg_temp.holdings_cusip_year h
    LEFT JOIN {ASSET_INDUSTRY} a ON a.cusip = h.cusip9
    GROUP BY ROLLUP (h.year)
    ORDER BY h.year NULLS LAST;
""", note=f"FF{CHECK_SCHEME}. value_not_in_asset_pct: holdings whose CUSIP is not in the asset table at all "
          "(no industry possible)")

chk.run("industry_distribution_holdings", f"""
    SELECT coalesce(a.ff::text, '(none)') AS ff, coalesce(a.ff_short, '(unclassified)') AS ff_short,
           sum(h.rows) AS rows,
           round(100.0 * sum(h.value) / sum(sum(h.value)) OVER (), 2) AS value_pct
    FROM pg_temp.holdings_cusip_year h
    LEFT JOIN {ASSET_INDUSTRY} a ON a.cusip = h.cusip9
    GROUP BY 1, 2
    ORDER BY value_pct DESC;
""", note=f"FF{CHECK_SCHEME}")

chk.run("top_unclassified_holdings", f"""
    -- largest unclassified CUSIPs by holdings value: candidates for manual fixes
    SELECT h.cusip9, a.issuer_name, a.security_type2, a.figi_ticker, a.industry_match, a.sic,
           sum(h.rows) AS rows, round(sum(h.value)) AS value
    FROM pg_temp.holdings_cusip_year h
    LEFT JOIN {ASSET_INDUSTRY} a ON a.cusip = h.cusip9
    WHERE a.ff IS NULL
    GROUP BY 1, 2, 3, 4, 5, 6
    ORDER BY value DESC
    LIMIT 40;
""", note=f"FF{CHECK_SCHEME}")

✅ **[industry_checks] holdings_per_cusip_year**: `ok` | `SELECT 186331` | rows: **186,331** | 88.0s

> NOTICES: NOTICE:  schema "pg_temp" does not exist, skipping

✅ **[industry_checks] scheme_coverage**: `ok` | `SELECT 8` | rows: **8** | 0.3s

> schemes with a catch-all industry (5, 10, 12, 38) classify every holding whose issuer has a SIC

,scheme,rows_pct,value_pct
0,5,87.7,92.0
1,10,87.7,92.0
2,12,87.7,92.0
3,17,86.9,91.5
4,30,87.0,91.5
5,38,87.7,92.0
6,48,87.5,91.9
7,49,87.5,91.9


...

,ff,ff_short,rows,value_pct
0,34,BusSv,8688760,11.45
1,(none),(unclassified),12187568,8.06
2,44,Banks,6338602,7.45
3,13,Drugs,5451891,7.29
4,30,Oil,4756323,6.05
5,42,Rtail,4772835,5.73
6,47,Fin,5964549,5.22
7,45,Insur,3834668,4.76
8,35,Comps,2696221,4.47
9,36,Chips,4620429,4.37


✅ **[industry_checks] top_unclassified_holdings**: `ok` | `SELECT 40` | rows: **40** | 0.1s

> FF48

,cusip9,issuer_name,security_type2,figi_ticker,industry_match,sic,rows,value
0,78462F103,SPDR S&P 500 ETF TR,Mutual Fund,SPY,unmatched,None,128991,7568779183
1,464287200,ISHARES TR,Mutual Fund,IVV,unmatched,None,62779,2553545737
2,464287465,ISHARES TR,Mutual Fund,EFA,unmatched,None,78193,2061973655
3,263534109,DU PONT E I DE NEMOURS & CO,None,None,unmatched,None,107629,2054635779
4,464287655,ISHARES TR,Mutual Fund,IWM,unmatched,None,71824,1903691932
5,902124106,TYCO INTL LTD NEW,None,None,ambiguous,None,41579,1607010671
6,984332106,YAHOO INC,None,None,unmatched,None,56481,1468314161
7,922042858,VANGUARD INTL EQUITY INDEX F,Mutual Fund,VWO,unmatched,None,59809,1345405584
8,464287234,ISHARES TR,Mutual Fund,EEM,unmatched,None,65551,1248797131
9,464287226,ISHARES TR,Mutual Fund,AGG,unmatched,None,40418,1168280535


,cusip9,issuer_name,security_type2,figi_ticker,industry_match,sic,rows,value
0,78462F103,SPDR S&P 500 ETF TR,Mutual Fund,SPY,unmatched,None,128991,7568779183
1,464287200,ISHARES TR,Mutual Fund,IVV,unmatched,None,62779,2553545737
2,464287465,ISHARES TR,Mutual Fund,EFA,unmatched,None,78193,2061973655
3,263534109,DU PONT E I DE NEMOURS & CO,None,None,unmatched,None,107629,2054635779
4,464287655,ISHARES TR,Mutual Fund,IWM,unmatched,None,71824,1903691932
5,902124106,TYCO INTL LTD NEW,None,None,ambiguous,None,41579,1607010671
6,984332106,YAHOO INC,None,None,unmatched,None,56481,1468314161
7,922042858,VANGUARD INTL EQUITY INDEX F,Mutual Fund,VWO,unmatched,None,59809,1345405584
8,464287234,ISHARES TR,Mutual Fund,EEM,unmatched,None,65551,1248797131
9,464287226,ISHARES TR,Mutual Fund,AGG,unmatched,None,40418,1168280535


### Unclassified assets (limitation, not fixed)

An asset gets an industry only through `issuer_cik`, which is NULL when the issuer matching (section 3) found no
EDGAR company for it:

| `industry_match` | meaning | assets (of 40,593) |
|---|---|---|
| `unmatched` | no EDGAR company fits the name | 8,251 (2,407 of them funds / ETFs) |
| `ambiguous` | several EDGAR companies fit, none is chosen | 662 |

* **Funds and ETFs** (SPY, IVV, IWM, …) have no SIC code in EDGAR, so they stay unclassified even with a CIK.
  They are about 5.5% of the holdings value.
* **Companies** without a match are about 3% of the holdings value. `top_unclassified_holdings` above lists the
  largest: some are large issuers whose 13(f) list name differs too much from the EDGAR name (DuPont, Yahoo,
  D R Horton), others are genuinely ambiguous and left unassigned on purpose rather than guessed (Gannett: old
  Gannett and the post-2015 Gannett / TEGNA; Tyco: several spin-offs).
* A manual CUSIP → CIK override for the largest of them was considered and not done: it cannot cover all issuers,
  and the residual share is small. These holdings stay in the data and the graph, only without an industry, so
  industry-level results slightly under-represent the affected industries.

Value shares from `graph_db` built 2026-10-14 (before the option rows were removed); the checks above give the
current figures.

### Ticker conflicts and ambiguous matches

`ticker_conflicts`: the OpenFIGI ticker belongs in EDGAR to another filer than the name match. Expected for
reused tickers and bonds of acquired firms; the name match is kept.

In [14]:
chk.run("ticker_conflicts", f"""
    SELECT m.cusip, a.issuer_name, a.figi_ticker, c.name AS matched_name, t.name AS ticker_owner,
           a.last_year
    FROM {MATCH_TABLE} m
    JOIN {ASSET_TABLE}   a ON a.cusip = m.cusip
    JOIN {COMPANY_TABLE} c ON c.cik = m.cik
    JOIN {COMPANY_TABLE} t ON t.cik = m.ticker_cik
    WHERE m.ticker_cik <> m.cik
    ORDER BY random()
    LIMIT 30;
""")

chk.run("ambiguous_examples", f"""
    SELECT m.cusip, a.issuer_name, a.first_year, a.last_year,
           string_agg(c.name || ' (' || coalesce(c.sic, '-') || ' / ' || coalesce(i.short_name, '-') || ')', ' | ')
               AS candidates
    FROM {MATCH_TABLE} m
    JOIN {ASSET_TABLE} a ON a.cusip = m.cusip
    CROSS JOIN LATERAL unnest(m.candidate_ciks) AS k(cik)
    JOIN {COMPANY_TABLE} c ON c.cik = k.cik
    LEFT JOIN {SIC_FF_TABLE}      f ON f.sic = c.sic AND f.scheme = {CHECK_SCHEME}
    LEFT JOIN {FF_INDUSTRY_TABLE} i ON i.scheme = f.scheme AND i.industry_number = f.industry_number
    WHERE m.method = 'ambiguous'
    GROUP BY 1, 2, 3, 4
    ORDER BY random()
    LIMIT 30;
""")

✅ **[industry_checks] ticker_conflicts**: `ok` | `SELECT 30` | rows: **30** | 0.0s

,cusip,issuer_name,figi_ticker,matched_name,ticker_owner,last_year
0,65333BAC0,NEXSTAR PHARMACEUTICALS INC,GILD 6.25 08/01/04,NEXSTAR PHARMACEUTICALS INC,"GILEAD SCIENCES, INC.",1999
1,590188A65,MERRILL LYNCH & CO INC,BAC 0 05/23/31,"MERRILL LYNCH & CO., INC.",BANK OF AMERICA CORP /DE/,2008
2,500440AA0,KOLLMORGEN CORP,DHR 8.75 05/01/09,KOLLMORGEN CORP,DANAHER CORP /DE/,2000
3,068055300,BARNETT BKS INC,BAC 4.5 PERP A,BARNETT BANKS INC,BANK OF AMERICA CORP /DE/,1996
4,316326AA5,FIDELITY NATL FINL INC,FIS 0 02/15/09,FIDELITY NATIONAL FINANCIAL INC /DE/,"Fidelity National Information Services, Inc.",1999
5,032511AP2,ANADARKO PETE CORP,APC 0 03/07/20,ANADARKO PETROLEUM CORP,ARKO Petroleum Corp.,2003
6,475070207,JEFFERSON PILOT CORP,LNC 7.25 01/21/00,JEFFERSON PILOT CORP,LINCOLN NATIONAL CORP,1997
7,17324XB90,CITIGROUP GLOBAL MKTS HLDGS,C 0 05/08/25 GMU2,Citigroup Global Markets Holdings Inc.,CITIGROUP INC,2020
8,17324XYF1,CITIGROUP GLOBAL MKTS HLDGS,C V0 05/11/23 DBTN,Citigroup Global Markets Holdings Inc.,CITIGROUP INC,2020
9,025932104,AMERICAN FINL GROUP INC OHIO,AFG,AMERICAN FINANCIAL GROUP INC ET AL,AMERICAN FINANCIAL GROUP INC,2020


✅ **[industry_checks] ambiguous_examples**: `ok` | `SELECT 30` | rows: **30** | 0.0s

,cusip,issuer_name,first_year,last_year,candidates
0,144418100,CARRAMERICA RLTY CORP,1998,2006,CARRAMERICA REALTY L P (6500 / RlEst) | CARRAM...
1,319779401,FIRST COMM CORP,1996,1997,FIRST COMMERCE CORP (6021 / Banks) | FIRST COM...
2,000915306,ADT LTD,1996,1997,Johnson Controls International plc (3585 / Mac...
3,872876107,TSI INC MINN,1996,2000,TSI INC /MN/ (3823 / LabEq) | TSI INC /MT/ (63...
4,670851302,OI S.A.,2014,2016,OI S.A. - In Judicial Reorganization (4813 / T...
5,61844A117,MORRISON KNUDSEN CORP NEW,1996,2000,MORRISON KNUDSEN CORP (1540 / Cnstr) | WASHING...
6,866762107,SUN INC,1996,1998,SUN (7373 / Comps) | SUNOCO INC (2911 / Oil)
7,887315AQ2,TIME WARNER INC,1996,1996,"WARNER MEDIA, LLC (4841 / Telcm) | HISTORIC TW..."
8,92851S204,VIVENDI,2000,2006,VIVENDI (4813 / Telcm) | VIVENDI SA (4953 / Ot...
9,20009T147,COMBIMATRIX CORPORATION,2016,2017,COMBIMATRIX CORP (3826 / LabEq) | CombiMatrix ...


,cusip,issuer_name,first_year,last_year,candidates
0,144418100,CARRAMERICA RLTY CORP,1998,2006,CARRAMERICA REALTY L P (6500 / RlEst) | CARRAM...
1,319779401,FIRST COMM CORP,1996,1997,FIRST COMMERCE CORP (6021 / Banks) | FIRST COM...
2,000915306,ADT LTD,1996,1997,Johnson Controls International plc (3585 / Mac...
3,872876107,TSI INC MINN,1996,2000,TSI INC /MN/ (3823 / LabEq) | TSI INC /MT/ (63...
4,670851302,OI S.A.,2014,2016,OI S.A. - In Judicial Reorganization (4813 / T...
5,61844A117,MORRISON KNUDSEN CORP NEW,1996,2000,MORRISON KNUDSEN CORP (1540 / Cnstr) | WASHING...
6,866762107,SUN INC,1996,1998,SUN (7373 / Comps) | SUNOCO INC (2911 / Oil)
7,887315AQ2,TIME WARNER INC,1996,1996,"WARNER MEDIA, LLC (4841 / Telcm) | HISTORIC TW..."
8,92851S204,VIVENDI,2000,2006,VIVENDI (4813 / Telcm) | VIVENDI SA (4953 / Ot...
9,20009T147,COMBIMATRIX CORPORATION,2016,2017,COMBIMATRIX CORP (3826 / LabEq) | CombiMatrix ...


### Manual review sample

Random name matches where the EDGAR **current** name differs from the issuer name (renames and abbreviations).
Check a few by eye after every change of the matching rules.

In [15]:
chk.run("review_sample", f"""
    SELECT a.cusip, a.issuer_name, c.name AS edgar_name, a.sic, a.ff_short, a.industry_match
    FROM {ASSET_INDUSTRY} a
    JOIN {COMPANY_TABLE} c ON c.cik = a.issuer_cik
    WHERE upper(c.name) NOT LIKE upper(left(a.issuer_name, 10)) || '%'
    ORDER BY random()
    LIMIT 40;
""", note=f"ff_short: FF{CHECK_SCHEME}")

✅ **[industry_checks] review_sample**: `ok` | `SELECT 40` | rows: **40** | 0.0s

> ff_short: FF48

,cusip,issuer_name,edgar_name,sic,ff_short,industry_match
0,740327101,PRECISION STD INC,"ALABAMA AIRCRAFT INDUSTRIES, INC",3721,Aero,name
1,74348TAK8,PROSPECT CAPITAL CORPORATION,Prospector Capital Corp.,6770,Fin,name
2,76116A306,RESOLUTE ENERGY CORP,Cimarex Resolute LLC,1311,Oil,name
3,315662106,FIBERSTARS INC,"ENERGY FOCUS, INC/DE",3640,ElcEq,name
4,302374103,F & M NATL CORP,F&M NATIONAL CORP,6021,Banks,name
5,01642TAC2,ALKERMES INC,Alkermes plc.,2834,Drugs,name+ticker
6,948626106,WEIGHT WATCHERS INTL INC NEW,"WW INTERNATIONAL, INC.",7200,PerSv,name
7,219023108,CORN PRODS INTL INC,Ingredion Inc,2040,Food,name
8,92656G108,VIDEO JUKEBOX NETWORK INC,BOX WORLDWIDE INC,4833,Telcm,name
9,G8018D107,SECURITY CAPITAL ASSURANCE,SYNCORA HOLDINGS LTD,6351,Insur,name


,cusip,issuer_name,edgar_name,sic,ff_short,industry_match
0,740327101,PRECISION STD INC,"ALABAMA AIRCRAFT INDUSTRIES, INC",3721,Aero,name
1,74348TAK8,PROSPECT CAPITAL CORPORATION,Prospector Capital Corp.,6770,Fin,name
2,76116A306,RESOLUTE ENERGY CORP,Cimarex Resolute LLC,1311,Oil,name
3,315662106,FIBERSTARS INC,"ENERGY FOCUS, INC/DE",3640,ElcEq,name
4,302374103,F & M NATL CORP,F&M NATIONAL CORP,6021,Banks,name
5,01642TAC2,ALKERMES INC,Alkermes plc.,2834,Drugs,name+ticker
6,948626106,WEIGHT WATCHERS INTL INC NEW,"WW INTERNATIONAL, INC.",7200,PerSv,name
7,219023108,CORN PRODS INTL INC,Ingredion Inc,2040,Food,name
8,92656G108,VIDEO JUKEBOX NETWORK INC,BOX WORLDWIDE INC,4833,Telcm,name
9,G8018D107,SECURITY CAPITAL ASSURANCE,SYNCORA HOLDINGS LTD,6351,Insur,name


In [16]:
chk.table_size(FF_INDUSTRY_TABLE, "ff_industry")
chk.table_size(SIC_FF_TABLE, "sic_ff")
chk.table_size(MATCH_TABLE, "match")

✅ **[industry_checks] ff_industry_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,total,heap,indexes
0,64 kB,16 kB,16 kB


✅ **[industry_checks] sic_ff_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,total,heap,indexes
0,4912 kB,2584 kB,2296 kB


✅ **[industry_checks] match_size**: `ok` | `SELECT 1` | rows: **1** | 0.0s

,total,heap,indexes
0,4944 kB,3288 kB,1616 kB


,total,heap,indexes
0,4944 kB,3288 kB,1616 kB


## 5. Run log analytics

Reads `steps.jsonl` from **all** runs in `logs/industry/`.

In [17]:
logs = load_logs(LOG_ROOT)
steps_table(logs, p.run_id)

,seq,phase,step,status,statusmessage,rowcount,expected,diff,duration_s
66,1,connection,connection_check_server,ok,None,NaN,NaN,NaN,0.017
67,2,connection,connection_check_schemas,ok,None,NaN,NaN,NaN,NaN
68,3,connection,connection_check_tables,ok,None,NaN,NaN,NaN,NaN
69,4,industry_ff,create_ff_tables,ok,CREATE TABLE,-1.0,NaN,NaN,0.018
70,5,industry_ff,import_ff_industry,ok,COPY 209,209.0,209.0,0.0,0.003
71,6,industry_ff,import_sic_ff,ok,COPY 59627,59627.0,59627.0,0.0,0.201
72,7,industry_ff,idx_sic_ff_industry,ok,CREATE INDEX,-1.0,NaN,NaN,0.022
73,8,industry_ff,analyze_ff,ok,ANALYZE,-1.0,NaN,NaN,0.024
74,9,industry_ff,ff_schemes,ok,SELECT 8,8.0,NaN,NaN,0.004
75,10,industry_ff,ff_sanity,ok,SELECT 7,7.0,NaN,NaN,0.001


In [18]:
# Row counts that differ from the previous run
expected_mismatches(logs, p.run_id)

,seq,phase,step,status,statusmessage,rowcount,expected,diff,duration_s


In [19]:
phase_durations(logs, p.run_id)

,seconds
phase,
connection,0.017
industry_ff,0.274
industry_match,3.820
industry_asset,0.986
industry_checks,88.656


In [20]:
compare_runs(logs)

rowcount                                      duration_s                                
run_id                                         20261011_151833 20261011_201813 20261029_160418 20261011_151833 20261011_201813 20261029_160418
phase           step                                                                                                                          
connection      connection_check_server                    NaN             NaN             NaN           0.026           0.016           0.017
industry_ff     create_ff_tables                          -1.0            -1.0            -1.0           0.019           0.016           0.018
                import_ff_industries                     209.0             NaN             NaN           0.002             NaN             NaN
                import_sic_to_ff                       10000.0             NaN             NaN           0.028             NaN             NaN
                ff_schemes                                 8.0             8.0             8.0           0.022           0.004           0.004
                ff_lookup_unknown_industries               0.0             NaN             NaN           0.011             NaN             NaN
                create_ff48_of                            -1.0             NaN             NaN           0.009             NaN             NaN
                ff48_sanity                                1.0             NaN             NaN           0.001             NaN             NaN
                import_ff_industry                         NaN           209.0           209.0             NaN           0.001           0.003
                import_sic_ff                              NaN         59627.0         59627.0             NaN           0.217           0.201
                idx_sic_ff_industry                        NaN            -1.0            -1.0             NaN           0.022           0.022
                analyze_ff                                 NaN            -1.0            -1.0             NaN           0.027           0.024
                ff_sanity                                  NaN             7.0             7.0             NaN           0.001           0.001
                drop_sic_to_ff                             NaN            -1.0            -1.0             NaN           0.006           0.001
                drop_ff_industries                         NaN            -1.0            -1.0             NaN           0.002           0.000
                drop_ff48_of                               NaN            -1.0            -1.0             NaN           0.002           0.000
industry_match  create_match_table                        -1.0            -1.0            -1.0           0.014           0.009           0.010
                match_issuers                          40593.0         40593.0         40593.0           3.867           3.896           3.791
                match_count_exact                          1.0             1.0             1.0           0.003           0.002           0.002
                match_by_security_type                    12.0            12.0            12.0           0.021           0.014           0.017
industry_asset  add_industry_columns                      -1.0             NaN             NaN           0.004             NaN             NaN
                reset_industry_columns                 67216.0             NaN             NaN           0.778             NaN             NaN
                update_underlying                      40593.0             NaN             NaN           0.672             NaN             NaN
                update_options                         26623.0             NaN             NaN           0.318             NaN             NaN
                idx_asset_ff48                            -1.0             NaN             NaN           0.031             NaN             NaN
                analyze_asset                     

In [21]:
p.close()

Run log: <THESIS_DIR>/src/Clean_solution/Database/Initialisation/logs/industry/20261029_160418/steps.jsonl
